# LangSmith Studio

在本地用 LangChain 构建 agent 时, 能可视化 agent 内部正在发生什么、与它实时交互, 并在问题出现的当下进行调试, 会很有帮助。**LangSmith Studio** 是一个免费的可视化界面, 让你能在本地机器上开发和测试你的 LangChain agent。

Studio 会连接你在本地运行的 agent, 向你展示 agent 所采取的每一步: 发送给 model 的 prompt、tool call 及其结果, 以及最终输出。你可以测试不同的输入、检查中间 state, 并在无需额外代码或部署的情况下迭代 agent 的行为。

本页介绍如何为你的本地 LangChain agent 设置 Studio。

## 前置条件

在开始之前, 请确保你具备以下条件:

- **一个 LangSmith 账号**: 在 [smith.langchain.com](https://smith.langchain.com) 注册或登录。
- **一个 LangSmith API key**: 按照[创建 API key](https://docs.langchain.com/langsmith/create-account-api-key) 指南操作。

## 设置本地 Agent server

### 1. 安装 LangGraph CLI

[LangGraph CLI](https://docs.langchain.com/langsmith/cli) 提供一个本地开发服务器 (也称为 [Agent Server](https://docs.langchain.com/langsmith/agent-server)), 它把你的 agent 连接到 Studio。

```shell
# Python >= 3.11 is required.
pip install --upgrade "langgraph-cli[inmem]"
```

### 2. 准备你的 agent

如果你已经有一个 LangChain agent, 可以直接使用它。本示例使用一个简单的 email agent:

In [1]:
# 环境准备: 从 .env 加载 API keys
from dotenv import load_dotenv

load_dotenv()  # 自动向上查找 src/oss/langgraph/.env

# ---- 统一导入 ----
from langchain.agents import create_agent

In [2]:
def send_email(to: str, subject: str, body: str):
    """Send an email"""
    email = {
        "to": to,
        "subject": subject,
        "body": body
    }
    # ... email sending logic

    return f"Email sent to {to}"


agent = create_agent(
    "deepseek:deepseek-chat",
    tools=[send_email],
    system_prompt="You are an email assistant. Always use the send_email tool.",
)

### 3. 环境变量

Studio 需要一个 LangSmith API key 来连接你的本地 agent。在项目根目录创建一个 `.env` 文件, 并添加你从 [LangSmith](https://smith.langchain.com/settings) 获取的 API key。

> **注意**
>
> 确保你的 `.env` 文件没有被提交到版本控制系统, 例如 Git。

```bash
LANGSMITH_API_KEY=lsv2...
```

### 4. 创建 LangGraph 配置文件

LangGraph CLI 使用一个配置文件来定位你的 agent 并管理依赖。在你的应用目录中创建一个 `langgraph.json` 文件:

```json
{
  "dependencies": ["."],
  "graphs": {
    "agent": "./src/agent.py:agent"
  },
  "env": ".env"
}
```

`create_agent` 函数会自动返回一个已编译的 LangGraph graph, 这正是配置文件中 `graphs` 键所期望的。

> **提示**
>
> 关于配置文件的 JSON 对象中每个键的详细说明, 请参阅 [LangGraph 配置文件参考](https://docs.langchain.com/langsmith/cli#configuration-file)。

此时, 项目结构将如下所示:

```bash
my-app/
├── src
│   └── agent.py
├── .env
└── langgraph.json
```

### 5. 安装依赖

从根目录安装项目依赖:

```bash
pip install langchain langchain-deepseek
```

```bash
uv add langchain langchain-deepseek
```

### 6. 在 Studio 中查看你的 agent

启动开发服务器, 把你的 agent 连接到 Studio:

```shell
langgraph dev
```

> **注意**
>
> Safari 会阻止到 Studio 的 `localhost` 连接。要绕过这个问题, 请用 `--tunnel` 运行上述命令, 以便通过安全隧道访问 Studio。你需要手动把隧道 URL 添加到允许的来源: 在 Studio UI 中点击 **Connect to a local server**。具体步骤见[故障排查指南](https://docs.langchain.com/langsmith/troubleshooting-studio#safari-connection-issues)。

服务器运行后, 你的 agent 既可以通过 API 在 `http://127.0.0.1:2024` 访问, 也可以通过 Studio UI 在 `https://smith.langchain.com/studio/?baseUrl=http://127.0.0.1:2024` 访问。

将 Studio 连接到你的本地 agent 后, 你就可以快速迭代 agent 的行为。运行一个测试输入, 在 [LangSmith](https://docs.langchain.com/langsmith/observability-studio) 中检查完整的执行 trace, 其中包含 prompt、tool 参数、返回值以及 token/延迟指标。当出现问题时, Studio 会连同周围的 state 一起捕获异常, 帮助你理解发生了什么。

开发服务器支持热重载——在代码中修改 prompt 或 tool 签名, Studio 会立即反映这些更改。可以从任意步骤重新运行对话 thread, 以测试你的更改, 而无需从头开始。这套 workflow 既能适用于简单的单 tool agent, 也能适用于复杂的多 node graph。

关于如何运行 Studio 的更多信息, 请参阅 [LangSmith 文档](https://docs.langchain.com/langsmith/observability) 中的以下指南:

- [运行应用](https://docs.langchain.com/langsmith/use-studio#run-application)
- [管理 assistant](https://docs.langchain.com/langsmith/use-studio#manage-assistants)
- [管理 thread](https://docs.langchain.com/langsmith/use-studio#manage-threads)
- [迭代 prompt](https://docs.langchain.com/langsmith/observability-studio)
- [调试 LangSmith trace](https://docs.langchain.com/langsmith/observability-studio#debug-langsmith-traces)
- [向数据集添加 node](https://docs.langchain.com/langsmith/observability-studio#add-node-to-dataset)